# Iris KNN Classification

Objective: Develop a classification model using the K-Nearest Neighbors (KNN) algorithm to predict the species of iris flowers based on the Iris dataset.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit, LeaveOneOut, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Data Preprocessing

In [ ]:
# Load the dataset
df = pd.read_csv('iris_with_missing.csv')
print(df.head())

In [ ]:
# Describe the dataset
print(df.info())
print(df.describe())

In [ ]:
# Check for missing values
print(df.isnull().sum())

In [ ]:
# Handle missing values
imputer_num = SimpleImputer(strategy='mean')
imputer_cat = SimpleImputer(strategy='most_frequent')
numerical_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
categorical_cols = ['species']
df[numerical_cols] = imputer_num.fit_transform(df[numerical_cols])
df[categorical_cols] = imputer_cat.fit_transform(df[categorical_cols])

In [ ]:
# Encode categorical variables
le = LabelEncoder()
df['species'] = le.fit_transform(df['species'])

In [ ]:
# Feature scaling
scaler = StandardScaler()
X = df.drop('species', axis=1)
y = df['species']
X_scaled = scaler.fit_transform(X)

## Data Splitting

In [ ]:
# Stratified split: 70% train, 15% val, 15% test
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
for train_val_index, test_index in sss.split(X_scaled, y):
    X_train_val = X_scaled[train_val_index]
    y_train_val = y[train_val_index]
    X_test = X_scaled[test_index]
    y_test = y[test_index]

# Split train_val into train and val
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=15/85, random_state=42)
for train_index, val_index in sss2.split(X_train_val, y_train_val):
    X_train = X_train_val[train_index]
    y_train = y_train_val[train_index]
    X_val = X_train_val[val_index]
    y_val = y_train_val[val_index]

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

## Model Training & Hyperparameter Tuning

In [ ]:
# Train KNN with k=3
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

In [ ]:
# Tune hyperparameters on validation set
k_values = [3, 5, 7]
metrics = ['euclidean', 'manhattan']
best_score = 0
best_params = {}
for k in k_values:
    for metric in metrics:
        knn_temp = KNeighborsClassifier(n_neighbors=k, metric=metric)
        knn_temp.fit(X_train, y_train)
        score = knn_temp.score(X_val, y_val)
        if score > best_score:
            best_score = score
            best_params = {'k': k, 'metric': metric}
print(f"Best params: {best_params}, Score: {best_score}")

# Train final model with best params
knn_final = KNeighborsClassifier(n_neighbors=best_params['k'], metric=best_params['metric'])
knn_final.fit(X_train, y_train)

## Model Evaluation

In [ ]:
# Evaluate on test set
y_pred = knn_final.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(f"Precision: {precision_score(y_test, y_pred, average='macro')}")
print(f"Recall: {recall_score(y_test, y_pred, average='macro')}")
print(f"F1: {f1_score(y_test, y_pred, average='macro')}")

# ROC-AUC
y_pred_proba = knn_final.predict_proba(X_test)
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba, multi_class='ovr')}")

In [ ]:
# Leave-one-out cross-validation
loo = LeaveOneOut()
scores = cross_val_score(knn_final, X_scaled, y, cv=loo, scoring='accuracy')
print(f"LOO CV Accuracy: {scores.mean()}")

## Insights and Discussion

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Discussion: The KNN model achieved good performance on the Iris dataset. The best hyperparameters were found through validation. LOO CV confirms robustness.